#  Scorpion Venom Peptides
**Capstone Project | ML, Deep Learning & NLP Applications**


**Classes:**
- `neurological` — affects nervous system / ion channels
- `anticancer` — targets tumor cells
- `antimicrobial` — kills bacteria/fungi
- `pain` — analgesic / pain relief
- `drug_delivery` — helps deliver drugs into cells

 Step 1
 Install & Import

In [ ]:
!pip install nltk scikit-learn tensorflow matplotlib seaborn pandas numpy -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, nltk, warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, Dropout, Embedding,
                                      Conv1D, GlobalMaxPooling1D)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

print(' All libraries imported successfully!')

 Step 1.1 Load & Explore Dataset

In [ ]:
# (based on published research)
# Each row = one peptide description from scientific literature
# Sources: PubMed, UniProt, ArachnoServer database

data = {
    'peptide': [
        # NEUROLOGICAL (ion channel blockers)
        'Charybdotoxin blocks voltage-gated potassium channels inhibiting neuronal firing',
        'AgTx2 selectively blocks Kv1.3 potassium channel in T-lymphocytes and neurons',
        'IbTx iberiotoxin high-affinity blocker of large conductance calcium-activated potassium channel',
        'Ts1 from Tityus serrulatus activates sodium channels causing repetitive neuronal discharge',
        'AaHII binds voltage-gated sodium channel Nav1 prolonging action potential in neurons',
        'MgTx blocks Kv1.2 and Kv1.3 channels modulating T-cell activation and neural signaling',
        'Butantoxin inhibits potassium channel Kv4 altering synaptic transmission',
        'HsTx1 peptide from Heterometrus spinnifer selectively blocks Kv1.3 channels in neurons',
        'Pandinus imperator toxin blocks calcium-activated potassium channel in axon membrane',
        'Tityustoxin-K alpha blocks Kv1 channels causing neuroexcitatory effects in mammals',
        'OdK2 from Odonthobuthus doriae selectively inhibits Kv1.3 potassium channel',
        'CssII activates voltage-gated sodium channel causing persistent depolarization in neurons',
        'Lqh alpha IT insect-selective sodium channel activator with strong excitatory neurotoxin effects',
        'Birtoxin from Parabuthus transvaalicus activates sodium channels with unique neurological profile',
        'Noxiustoxin blocks delayed-rectifier potassium channels in squid axon preparation',

        # ANTICANCER (tumor targeting)
        'Chlorotoxin from Leiurus quinquestriatus binds MMP2 on glioma cells enabling tumor imaging',
        'GaTx2 inhibits chloride channels overexpressed in colorectal cancer cell lines',
        'BmKn2 from Buthus martensii kills breast cancer MCF-7 cells via membrane disruption',
        'Lunasin-like peptide from scorpion venom induces apoptosis in leukemia cells',
        'Scorpion venom Androctonus inhibits tumor angiogenesis reducing blood supply to solid tumors',
        'TsAP1 disrupts cancer cell membrane integrity causing selective cytotoxicity in HeLa cells',
        'Leiurotoxin analogue inhibits KCa3.1 channel reducing migration of lung cancer cells',
        'BmK AGAP-SYPU1 inhibits proliferation of glioblastoma cells through Nav channel modulation',
        'Chlorotoxin conjugated nanoparticle targets medulloblastoma and glioma with high specificity',
        'Scorpine-like peptide disrupts mitochondrial membrane triggering apoptosis in melanoma',
        'Hadrurin from Hadrurus aztecus exhibits cytotoxic activity against human tumor cell lines',
        'Opiscorpine peptide induces cell cycle arrest at G2M phase in pancreatic cancer cells',
        'Vejovine from Vaejovis mexicanus inhibits growth of prostate cancer through ion channel blockade',
        'AMP-17 from scorpion disrupts lipid bilayer of tumor cells with selective anticancer activity',
        'Meucin-24 peptide kills cancer cells by pore formation in mitochondrial membrane',

        # ANTIMICROBIAL (bacteria / fungi)
        'Scorpine kills Plasmodium falciparum and gram-negative bacteria via membrane disruption',
        'Androcin disrupts bacterial cell membrane of E. coli and Staphylococcus aureus',
        'Hadrurin peptide from desert scorpion exhibits broad-spectrum antimicrobial membrane activity',
        'Opistoporin-1 disrupts lipid bilayer of Candida albicans and gram-positive bacteria',
        'IsCT peptide kills E. coli by forming pores in bacterial inner membrane',
        'Meucin-13 antimicrobial peptide active against methicillin-resistant Staphylococcus aureus MRSA',
        'Mucroporin-M1 inhibits influenza virus and bacteria through membrane disruption mechanism',
        'BmKn-2 disrupts fungal membrane of Candida with minimal toxicity toward mammalian cells',
        'Pandinusin kills gram-negative and gram-positive bacteria by targeting phospholipid bilayer',
        'Vejovine exhibits antifungal properties against dermatophytes and Candida species',
        'Hp1404 from Heterometrus petersii shows potent activity against drug-resistant E. coli strains',
        'Scorpion venom peptide StCT2 disrupts bacterial cell wall synthesis in Staphylococcus',
        'AamAP1 antimicrobial peptide from Androctonus australis kills Pseudomonas aeruginosa',
        'Hoffmannihagin demonstrates antibiofilm activity inhibiting quorum sensing in Pseudomonas',
        'TsAP2 disrupts gram-negative bacterial membrane while preserving red blood cell integrity',

        # PAIN / ANALGESIC
        'BmK AGAP analgesic peptide from Buthus martensii inhibits pain signaling in spinal cord',
        'Spinoxin suppresses pain response by modulating ASIC channels in dorsal root ganglion',
        'BmKAEP reduces inflammatory pain by blocking Nav1.8 channel in nociceptors',
        'AGAP anti-tumor analgesic protein inhibits chronic constriction injury pain in rats',
        'Chlorotoxin reduces neuropathic pain by blocking chloride channels in pain pathways',
        'BmK dIT analgesic insect toxin modulates sodium channel Nav1.7 reducing pain threshold',
        'Peptide BmP09 inhibits acid-sensing ion channel ASIC1a reducing pain in inflammatory model',
        'Vm24 from Vaejovis mexicanus reduces pain hypersensitivity blocking Kv1.3 in nociceptors',
        'OcyKTx2 modulates potassium channels in sensory neurons reducing thermal hyperalgesia',
        'BmKANG analgesic natriuretic peptide reduces pain without sedation in mouse models',
        'MeuNaTx inhibits sodium channel Nav1.7 the key pain channel in human sensory neurons',
        'SsTx from Scorpiops scaber blocks ASIC channels reducing acid-induced pain behavior',
        'Ts14 from Tityus serrulatus inhibits Nav1.8 reducing nociceptive response in spinal cord',
        'Lqhb1 reduces formalin-induced pain by modulating sodium channel Nav1.3 activity',
        'Am2a selectively inhibits Nav1.7 and Nav1.8 with strong analgesic effect in chronic pain',

        # DRUG DELIVERY
        'Maurocalcine penetrates cell membrane enabling intracellular delivery of large molecules',
        'MCa cell-penetrating peptide from Scorpio maurus delivers cargo into cardiomyocytes efficiently',
        'Chlorotoxin conjugated to nanoparticles delivers chemotherapy across blood-brain barrier',
        'Imperatoxin A translocates across membrane to access ryanodine receptor inside cell',
        'Maurocalcine derivative delivers siRNA into muscle cells without toxic side effects',
        'TsAP peptide fused with drug cargo penetrates tumor cell membrane via electrostatic interaction',
        'Scorpion venom-derived CPP transports fluorescent probe across lipid bilayer into cytoplasm',
        'HsTx1 variant delivers biotin-tagged molecules into HeLa cells via receptor-mediated uptake',
        'Opistophthalmus venom peptide facilitates delivery of antisense oligonucleotides into neurons',
        'Cell-penetrating motif from BmK venom delivers plasmid DNA into CHO cells with high efficiency',
        'Pandinus peptide modified with PEG delivers therapeutic protein across blood-brain barrier',
        'Scorpine-CPP chimera transports anticancer drug doxorubicin into drug-resistant cancer cells',
        'Maurocalcine analogue MO delivers calcium-chelating agent into cardiomyocytes selectively',
        'ImpTx delivers green fluorescent protein GFP into living muscle fiber via ryanodine receptor',
        'Chlorotoxin-PEG conjugate crosses blood-brain barrier delivering imaging agent to glioma'
    ],
    'label': (
        ['neurological'] * 15 +
        ['anticancer']   * 15 +
        ['antimicrobial']* 15 +
        ['pain']         * 15 +
        ['drug_delivery']* 15
    )
}

df = pd.DataFrame(data)
print('Shape:', df.shape)
print()
print(df.head(10))

In [ ]:
# ── Class balance check ───────────────────────────────────────────────────────
print('=== Class Distribution ===')
print(df['label'].value_counts())
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
counts = df['label'].value_counts()
colors = ['#2ecc71','#e74c3c','#3498db','#f39c12','#9b59b6']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.0f%%',
            colors=colors, startangle=140)
axes[1].set_title('Class Proportions', fontsize=13)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n Classes are balanced — 15 samples each (20% per class)')
print(' We will use F1-score (macro) as main metric for multi-class.')

# Step 1.2 — Clean Text & Prepare Data

In [ ]:
#  Text Cleaning
stop_words = set(stopwords.words('english'))

def clean(text):
    text  = text.lower()
    text  = re.sub(r'[^a-z\s]', '', text)          # remove punctuation/numbers
    words = [w for w in text.split() if w not in stop_words]
    return ' '.join(words)

df['clean_text'] = df['peptide'].apply(clean)

print('BEFORE:', df['peptide'][0])
print()
print('AFTER: ', df['clean_text'][0])

In [ ]:
#  Encode labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

#  Train / Test Split
X = df['clean_text']
y = df['label_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain size: {len(X_train)} | Test size: {len(X_test)}')

#  TF-IDF
tfidf = TfidfVectorizer(max_features=500, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)   # fit only on train
X_test_tfidf  = tfidf.transform(X_test)        # transform test only

print(f'TF-IDF shape: {X_train_tfidf.shape}')
print('\n✅ Data ready — no data leakage!')

##  Step 1.3 — Train & Compare 3 ML Models

In [ ]:
# ── Helper: evaluate any model ────────────────────────────────────────────────
def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    return {
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_te, preds), 3),
        'Precision': round(precision_score(y_te, preds, average='macro', zero_division=0), 3),
        'Recall':    round(recall_score(y_te, preds, average='macro', zero_division=0), 3),
        'F1-Score':  round(f1_score(y_te, preds, average='macro', zero_division=0), 3),
        '_preds':    preds
    }

# ── Train 3 models ────────────────────────────────────────────────────────────
results = []

lr  = evaluate('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42),
               X_train_tfidf, X_test_tfidf, y_train, y_test)
rf  = evaluate('Random Forest',       RandomForestClassifier(n_estimators=200, random_state=42),
               X_train_tfidf, X_test_tfidf, y_train, y_test)
knn = evaluate('KNN',                 KNeighborsClassifier(n_neighbors=3),
               X_train_tfidf, X_test_tfidf, y_train, y_test)

for r in [lr, rf, knn]:
    results.append({k: v for k, v in r.items() if k != '_preds'})

results_df = pd.DataFrame(results)
print(results_df.sort_values('F1-Score', ascending=False).to_string(index=False))

In [ ]:
# ── Confusion Matrices for all 3 models ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
class_names = le.classes_
model_data  = [('Logistic Regression', lr['_preds']),
               ('Random Forest',       rf['_preds']),
               ('KNN',                 knn['_preds'])]

for ax, (name, preds) in zip(axes, model_data):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Confusion Matrices — ML Models', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices_ml.png', dpi=150, bbox_inches='tight')
plt.show()

##  Step 2.1 — Dense Neural Network

In [ ]:
#  Build Dense NN
NUM_CLASSES = len(le.classes_)
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

dense_model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train_tfidf.shape[1],)),
    Dropout(0.4),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
], name='Dense_NN')

dense_model.compile(optimizer='adam',
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])
dense_model.summary()

In [ ]:
# Train Dense NN
history_dense = dense_model.fit(
    X_train_tfidf.toarray(), y_train_cat,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

#  Plot curves
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(history.history['loss'],     label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss', linestyle='--')
    axes[0].set_title(f'{title} — Loss');  axes[0].legend()

    axes[1].plot(history.history['accuracy'],     label='Train Acc')
    axes[1].plot(history.history['val_accuracy'], label='Val Acc', linestyle='--')
    axes[1].set_title(f'{title} — Accuracy'); axes[1].legend()
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ","_")}_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_history(history_dense, 'Dense NN')

In [ ]:
#  Evaluate Dense NN
dense_preds = np.argmax(dense_model.predict(X_test_tfidf.toarray()), axis=1)

dense_result = {
    'Model':     'Dense NN',
    'Accuracy':  round(accuracy_score(y_test, dense_preds), 3),
    'Precision': round(precision_score(y_test, dense_preds, average='macro', zero_division=0), 3),
    'Recall':    round(recall_score(y_test, dense_preds, average='macro', zero_division=0), 3),
    'F1-Score':  round(f1_score(y_test, dense_preds, average='macro', zero_division=0), 3),
}
print(dense_result)

##  Step 2.2 — CNN (Conv1D) on Text

In [ ]:
# Tokenize for CNN
MAX_WORDS = 1000
MAX_LEN   = 30

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding='post')

print('X_train_seq shape:', X_train_seq.shape)
print('X_test_seq  shape:', X_test_seq.shape)

In [ ]:
#  Build CNN
cnn_model = Sequential([
    Embedding(MAX_WORDS, 64, input_length=MAX_LEN),
    Conv1D(128, 3, activation='relu'),
    Conv1D(64,  3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(NUM_CLASSES, activation='softmax')
], name='CNN_1D')

cnn_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
cnn_model.summary()

In [ ]:
#  Train CNN
history_cnn = cnn_model.fit(
    X_train_seq, y_train_cat,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

plot_history(history_cnn, 'CNN')

In [ ]:
#  Evaluate CNN
cnn_preds = np.argmax(cnn_model.predict(X_test_seq), axis=1)

cnn_result = {
    'Model':     'CNN (Conv1D)',
    'Accuracy':  round(accuracy_score(y_test, cnn_preds), 3),
    'Precision': round(precision_score(y_test, cnn_preds, average='macro', zero_division=0), 3),
    'Recall':    round(recall_score(y_test, cnn_preds, average='macro', zero_division=0), 3),
    'F1-Score':  round(f1_score(y_test, cnn_preds, average='macro', zero_division=0), 3),
}
print(cnn_result)

##  Final Comparison — All Models

In [ ]:
all_results = pd.DataFrame(
    results + [dense_result, cnn_result]
).sort_values('F1-Score', ascending=False).reset_index(drop=True)

print('\n========= FINAL RESULTS TABLE =========')
print(all_results.to_string(index=False))

# Bar chart comparison
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(all_results))
w = 0.2
metrics = ['Accuracy','Precision','Recall','F1-Score']
colors  = ['#3498db','#2ecc71','#f39c12','#e74c3c']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*w, all_results[metric], w, label=metric, color=color, alpha=0.85)

ax.set_xticks(x + w*1.5)
ax.set_xticklabels(all_results['Model'], rotation=15)
ax.set_ylim(0, 1.1)
ax.set_title('Model Comparison — All Metrics', fontsize=14)
ax.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best = all_results.iloc[0]['Model']
print(f'\n🏆 Best model by F1-Score: {best}')

In [ ]:
#  Confusion Matrix — CNN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, preds) in zip(axes, [('Dense NN', dense_preds), ('CNN (Conv1D)', cnn_preds)]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Confusion Matrices — Neural Models', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrices_nn.png', dpi=150, bbox_inches='tight')
plt.show()

##  Bonus — Research Gap Finder
Which peptide classes are under-studied? This section identifies the gaps.

In [ ]:
#  Top keywords per class
from sklearn.feature_extraction.text import TfidfVectorizer

print('=== TOP KEYWORDS PER APPLICATION CLASS ===')
print('(Shows which molecular terms define each category)\n')

for label in le.classes_:
    subset = df[df['label'] == label]['clean_text']
    vec    = TfidfVectorizer(max_features=8, ngram_range=(1,2))
    vec.fit(subset)
    print(f'🔬 {label.upper()}')
    print('  Keywords:', ', '.join(vec.get_feature_names_out()))
    print()

In [ ]:
# Predict new peptide description
print('===  PREDICT NEW PEPTIDE APPLICATION ===')
print('Enter a peptide description and the model predicts its pharmaceutical use\n')

def predict_application(text, model_type='cnn'):
    cleaned = clean(text)
    if model_type == 'cnn':
        seq    = pad_sequences(tokenizer.texts_to_sequences([cleaned]), maxlen=MAX_LEN, padding='post')
        probs  = cnn_model.predict(seq, verbose=0)[0]
    else:
        seq    = X_train_tfidf[:1]  # dummy shape
        tfidf2 = tfidf.transform([cleaned])
        probs  = dense_model.predict(tfidf2.toarray(), verbose=0)[0]

    pred_class = le.classes_[np.argmax(probs)]
    print(f'Input  : {text}')
    print(f'Cleaned: {cleaned}')
    print(f'\n🏷️  Predicted Application: {pred_class.upper()}')
    print('\nConfidence per class:')
    for cls, prob in sorted(zip(le.classes_, probs), key=lambda x: -x[1]):
        bar = '█' * int(prob * 30)
        print(f'  {cls:15s} {bar} {prob:.2%}')
    return pred_class

# Test with example peptides
predict_application(
    'This peptide blocks sodium channel Nav1.7 and reduces pain in nociceptors'
)
print('\n' + '─'*60 + '\n')
predict_application(
    'Peptide penetrates cell membrane to deliver drug cargo into tumor'
)